**EN ESTA FASE NO IMPORTA EL ARCHIVO EN SÍ, PORQUE ES EL MISMO QUE LA FASE 2, YA QUE EN LA FASE ANTERIOR SOLO ES DE EXPLORACIÓN INICIAL. 
PERO POR FAVOR CUANDO TERMINEN UNA FASE TRATEN DE DESCARGARLO COMO .CSV, ASÍ EN LA SIGUIENTE SE ABRE Y SE COMIENZA A TRABAJAR..
SI EN ESTA FASE SE LLAMA POR EJEMPLO df_datos_energia_limpio POR FAVOR EN LA SIGUIENTE, CUANDO ABRAN EL ARCHIVO COMO DF ponganle df_datos_energia_limpio 
SINO CUANDO UNAMOS LAS FASES NO VA A ANDAR!!!**

## Fase #3: Preparación de los Datos / Data Preparation
La fase de preparación de datos abarca todas las actividades para construir el conjunto de datos final (datos que se introducirán en las herramientas de modelado) a partir de los datos brutos iniciales. Es probable que las tareas de preparación de datos se realicen varias veces y no en ningún orden prescrito. Las tareas incluyen la selección de tablas, registros y atributos, así como la transformación y limpieza de datos para herramientas de modelado.
* Enumerá y describí las variables relevantes que son de tu interés para el estudio.Por ejemplo, edad: es la edad en años cumplidos al 31 de diciembre de 2024, es una variable continua discreta.
* Armá una tabla que resuma el análisis estadístico descriptivo y los gráficos que realizarás para entender las características de los datos.

### Variables seleccionadas

| Variable | Rol | Tipo | Definición operativa |
|---|---|---|---|
| `agua_inyectada_m3` | Objetivo | Cuantitativa continua | Volumen de agua registrado para la fractura, en m³. Debe ser mayor que cero. |
| `longitud_rama_horizontal_m` | Entrada | Cuantitativa continua | Longitud horizontal informada, en metros. |
| `cantidad_fracturas` | Entrada | Cuantitativa discreta | Cantidad de etapas o fracturas informadas para la operación. |
| `formacion_productiva` | Entrada | Cualitativa nominal | Formación productiva declarada. |
| `subtipo_reservorio` | Entrada | Cualitativa nominal | Subtipo no convencional, por ejemplo shale o tight. |
| `tipo_terminacion` | Entrada | Cualitativa nominal | Técnica de terminación informada. |
| `fecha_inicio_fractura` | Partición | Temporal | Inicio de la operación; se usa para separar entrenamiento y prueba. |
| `idpozo` | Control | Identificador | Permite impedir que un mismo pozo aparezca en ambos períodos. |

**Plan descriptivo.** Se calcularán conteo, faltantes, media, mediana, desvío y percentiles para las variables numéricas; frecuencia y volumen mediano por categoría; histograma de agua; dispersión entre longitud y agua; y comparación entre cantidad de etapas y agua. La escala logarítmica sólo se usará para visualizar, sin alterar los valores originales.

In [ ]:
# @title Imports
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display  # ty: ignore[unresolved-import]

from comun import leer_dataframe_con_fechas


In [ ]:
# @title Cargar datos
# Cargar el archivo CSV descargado en un DataFrame de pandas

df_datos_energia = leer_dataframe_con_fechas()
# Visualizar las primeras filas del DataFrame
df_datos_energia.head()

In [ ]:
variables_numericas = ["agua_inyectada_m3", "longitud_rama_horizontal_m", "cantidad_fracturas"]
variables_categoricas = ["formacion_productiva", "subtipo_reservorio", "tipo_terminacion"]

resumen_numerico = (
	df_datos_energia[variables_numericas]
	.describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.95])
	.T.rename(columns={"count": "n", "mean": "media", "std": "desvío", "min": "mínimo", "max": "máximo"})
)
display(resumen_numerico.round(2))

resumen_categorico = pd.concat(
	[
		df_datos_energia.groupby(col, dropna=False)["agua_inyectada_m3"]
		.agg(registros="size", agua_mediana_m3="median", agua_media_m3="mean")
		.assign(variable=col)
		.reset_index()
		.rename(columns={col: "categoría"})
		for col in variables_categoricas
	],
	ignore_index=True,
)
display(resumen_categorico.sort_values(["variable", "registros"], ascending=[True, False]).round(1))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(data=df_datos_energia, x="agua_inyectada_m3", bins=40, ax=axes[0])
axes[0].set(title="Distribución del agua inyectada", xlabel="Agua inyectada (m³)", ylabel="Registros")
axes[0].set_yscale("log")

sns.scatterplot(data=df_datos_energia, x="longitud_rama_horizontal_m", y="agua_inyectada_m3", alpha=0.45, s=25, ax=axes[1])
axes[1].set(title="Agua y longitud horizontal", xlabel="Longitud horizontal (m)", ylabel="Agua inyectada (m³)")

sns.scatterplot(data=df_datos_energia, x="cantidad_fracturas", y="agua_inyectada_m3", alpha=0.45, s=25, ax=axes[2])
axes[2].set(title="Agua y cantidad de fracturas", xlabel="Cantidad de fracturas", ylabel="Agua inyectada (m³)")
plt.tight_layout()
plt.show()

correlaciones = df_datos_energia[variables_numericas].corr(method="spearman")
display(correlaciones.round(3))


### Interpretación exploratoria

**Distribuciones numéricas.** El volumen de agua tiene una mediana de **56.829 m³** y una media de **57.272 m³**. La mitad central de las operaciones se ubica entre **38.509 y 75.540 m³**. El máximo de **537.185 m³** queda muy alejado del percentil 95 (**100.400 m³**) y debe revisarse como caso extremo, aunque no corresponde eliminarlo sin evidencia de error. La longitud horizontal mediana es **2.516 m** y la cantidad mediana es **40 fracturas**.

**Composición de la muestra.** Vaca Muerta concentra 2.527 de 2.626 registros (**96,2 %**) y el subtipo shale concentra 2.528 (**96,3 %**). También predomina la terminación Tapón disparo, con 2.094 registros (**79,7 %**). En consecuencia, el modelo queda especialmente orientado a operaciones shale de Vaca Muerta. Las categorías minoritarias se muestran como contexto y señalan oportunidades para ampliar la base en futuras versiones.

**Gráfico 1 — distribución del agua.** La mayor parte de los registros se concentra aproximadamente entre 20.000 y 100.000 m³. La barra aislada cercana a 537.000 m³ muestra unos pocos valores extremos. El eje vertical está en escala logarítmica para que esos grupos pequeños sean visibles; esto cambia la representación, no los datos.

**Gráfico 2 — agua y longitud horizontal.** La nube asciende: las ramas más largas tienden a registrar más agua. La relación no es exacta y la dispersión aumenta en longitudes altas, lo que indica que la longitud por sí sola no explica todo el volumen. Los dos puntos superiores a 500.000 m³ son atípicos frente al patrón general.

**Gráfico 3 — agua y cantidad de fracturas.** Se observa una relación ascendente todavía más definida: al aumentar la cantidad de etapas suele aumentar el agua registrada. Persisten variaciones entre operaciones con igual cantidad de fracturas y algunos casos extremos, por lo que esta variable aporta información importante pero no determina el resultado por sí sola.

**Correlaciones de Spearman.** Agua presenta una asociación monotónica fuerte con longitud horizontal (**0,833**) y especialmente con cantidad de fracturas (**0,913**). Longitud y cantidad de fracturas también están muy asociadas entre sí (**0,897**); esto significa que aportan información parcialmente redundante y puede repartir su importancia dentro del modelo. Estas correlaciones describen asociación, no causalidad.